## 3. Ajan ile Araç Kullanımı (Agent + Tools)
* Amaç: LangChain agent mimarisini öğretmek.
* Senaryo: Ajan, bir hesap makinesi veya arama motoru aracını nasıl ve ne zaman kullanacağına karar verir.
* Kazandırılan: Araç tanımı, dinamik karar verme.
* Kullanılan Bileşenler: initialize_agent, Tool, AgentType

In [ ]:
# ============================================================
# 1) KURULUM (Kaggle)
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               "langchain-experimental>=0.0.64" \
               transformers accelerate sentencepiece duckduckgo-search


In [ ]:

# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline

from langchain.agents import initialize_agent, AgentType
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# ============================================================
# 3) LLM: Açık kaynak model
#    Not: Qwen 0.5B ReAct uyumu TinyLlama'dan genelde daha iyi.
# ============================================================
# MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # önerilen

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1,
    top_p=0.9,
    repetition_penalty=1.05,
)
llm = HuggingFacePipeline(pipeline=gen_pipe)

# ============================================================
# 4) ARAÇLAR: Sadece DuckDuckGo (döngü ve parse hatalarını azaltır)
# ============================================================
ddg = DuckDuckGoSearchAPIWrapper(max_results=3)   # sonuçları sınırlayalım
search_tool = DuckDuckGoSearchRun(api_wrapper=ddg, name="WebSearch")
tools = [search_tool]

# ============================================================
# 5) AGENT: ZERO_SHOT_REACT_DESCRIPTION
#    - max_iterations: az tut
#    - early_stopping_method="force": Final Answer parse edilmeden durdur
# ============================================================
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=2,
    early_stopping_method="force"
)

# ============================================================
# 6) ÇALIŞTIR
# ============================================================
print("\n===== Örnek: Web araması =====\n")
question = "OpenAI kurucuları kimlerdir?"
answer = agent.run(question)
print("\n=== Cevap ===\n", answer)
